# RAG dev - parsing the handbook chapters

Scratch notebook for checking `retrieval/parse.py` by eye on the real FAA
Powerplant Handbook chapters. All the logic lives in
`src/industrial_agent/retrieval/`; this only calls it and prints.

## Setup

In [1]:
import os, sys

if not os.path.exists("repo"):
    !git clone -q -b feature/maintenance-docs-rag https://github.com/Cambie26/industrial_agent.git repo
%pip install -q -e "./repo[rag]"

sys.path.insert(0, os.path.abspath("repo/src"))
sys.path.insert(0, os.path.abspath("repo"))

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for industrial-agent (pyproject.toml) ... done


## Corpus

Upload one PDF per chapter to `repo/data/raw/`, named with the chapter number
(`ch10.pdf`). `data/` is gitignored, so they won't be committed. To read them
from Drive instead, mount it and point `RAW_DIR` at the folder.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path

RAW_DIR = Path("drive/MyDrive/faa_powerplant")

#RAW_DIR = Path("repo/data/raw")

In [4]:
RAW_DIR

PosixPath('drive/MyDrive/faa_powerplant')

In [5]:
assert RAW_DIR.is_dir(), f"Missing folder: {RAW_DIR}"
print(sorted(p.name for p in RAW_DIR.glob("*.pdf")))

['03_amtp_ch1.pdf', '04_amtp_ch2.pdf', '05_amtp_ch3.pdf', '06_amtp_ch4.pdf', '07_amtp_ch5.pdf', '08_amtp_ch6.pdf', '09_amtp_ch7.pdf', '10_amtp_ch8.pdf', '11_amtp_ch9.pdf', '12_amtp_ch10.pdf', '13_amtp_ch11.pdf']


In [ ]:
#!cd repo && python scripts/build_index.py --raw-dir /content/drive/MyDrive/faa_powerplant --index-dir /content/drive/MyDrive/faa_powerplant/index

ch10.pdf: 869 blocks -> 201 chunks
modules.json: 100% 349/349 [00:00<00:00, 1.02MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 469kB/s]
README.md: 100% 94.8k/94.8k [00:00<00:00, 72.6MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 212kB/s]
config.json: 100% 743/743 [00:00<00:00, 4.52MB/s]

model.safetensors: downloading bytes:  49% 64.9M/133M [00:01<00:00, 97.3MB/s, 4.33MB/s  ]
model.safetensors: reconstructing file:  50% 67.0M/133M [00:01<00:01, 58.8MB/s]
model.safetensors: downloading bytes: 100% 86.7M/86.7M [00:01<00:00, 54.0MB/s, 8.10MB/s  ]
model.safetensors: reconstructing file: 100% 133M/133M [00:01<00:00, 83.1MB/s, 12.7MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 7175.11it/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 2.11MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 57.9MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 120MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 450kB/s]
config.json: 100% 190/190 [00:00<00:00, 8

In [6]:
from collections import Counter

from industrial_agent.retrieval import parse


pdf_paths = parse.chapter_pdfs(RAW_DIR)
for pdf_path in pdf_paths:
    print(f"chapter {parse.chapter_number(pdf_path):>2}: {pdf_path.name}")

chapter  1: 03_amtp_ch1.pdf
chapter  2: 04_amtp_ch2.pdf
chapter  3: 05_amtp_ch3.pdf
chapter  4: 06_amtp_ch4.pdf
chapter  5: 07_amtp_ch5.pdf
chapter  6: 08_amtp_ch6.pdf
chapter  7: 09_amtp_ch7.pdf
chapter  8: 10_amtp_ch8.pdf
chapter  9: 11_amtp_ch9.pdf
chapter 10: 12_amtp_ch10.pdf
chapter 11: 13_amtp_ch11.pdf


## Parse one chapter

In [9]:
CHAPTER = 2
pdf_path = next(path for path in pdf_paths if parse.chapter_number(path) == CHAPTER)

blocks = parse.parse_pdf(pdf_path)
pages = {block.page for block in blocks}
print(f"{pdf_path.name}: {len(pages)} pages, {len(blocks)} blocks")
print(Counter(block.kind for block in blocks))

04_amtp_ch2.pdf: 46 pages, 499 blocks
Counter({'text': 354, 'heading': 81, 'caption': 64})


### Contents

Chunking takes its section boundaries from this. If it raises, the chapter
PDF has no bookmarks and the contents page will need parsing instead.

In [10]:
import inspect, dataclasses
print(inspect.getsource(parse.parse_pdf))
print(type(blocks), type(blocks[0]))
print(blocks[0])
if dataclasses.is_dataclass(blocks[0]):
    print([f.name for f in dataclasses.fields(blocks[0])])

def parse_pdf(pdf_path: Path | str, table_aware: bool = True) -> list[Block]:
    """Parse one chapter PDF into blocks, page by page.

    With `table_aware=False` each page is one text block from PyMuPDF's plain
    text extraction - the naive baseline for the ablation.
    """
    import pymupdf

    pdf_path = Path(pdf_path)
    if not pdf_path.exists():
        raise FileNotFoundError(f"No PDF at {pdf_path}.")
    chapter = chapter_number(pdf_path)

    blocks: list[Block] = []
    with pymupdf.open(pdf_path) as document:
        # Loop through each page, numbering from 1 to match the printed PDF.
        for page_index, page in enumerate(document):
            page_number = page_index + 1
            if table_aware:
                blocks.extend(parse_page(page, pdf_path.name, chapter, page_number))
                continue
            text = page.get_text("text").strip()
            if text:
                blocks.append(Block(pdf_path.name, chapter, page_number, "text", text))



In [11]:
for level, title, page in parse.table_of_contents(pdf_path):
    print(f"{'  ' * (level - 1)}{title}  (p.{page})")

Structure Bookmarks  (p.-1)
  Fuel System Requirements  (p.1)
    Fuel System Requirements  (p.1)
      Fuel System Requirements  (p.1)
        Fuel System Requirements  (p.1)
        The engine fuel system must supply fuel to the engine’s fuel metering device under all conditions of ground and air operation. It must function properly at constantly changing altitudes and in any climate. The most common fuels are AVGAS for reciprocating engines and Jet A for turbine engines. AVGAS is generally either 80 (red) or 100LL (blue) octane. The LL stands for low lead although it contains four times the lead of 80 octane AVGAS. Jet A is a kerosene-based fuel that is clear to straw in color.  (p.1)
        Electronic engine controls have allowed great increases in controlling the metered fuel flow to the engine. Engine fuel systems have become very accurate at providing the correct mixture of fuel and air to the engines. Gas turbine fuel controls have also greatly improved the ability to schedule

### Reading order on one page

Check the two columns come out left then right, not interleaved.

In [12]:
PAGE = 3
for block in blocks:
    if block.page == PAGE:
        print(f"[{block.kind}] {block.text[:200]}\n")

[caption] Figure 2-1. Air-fuel mixture curves.

[text] takeoff speed. [Figure 2-2] Since it is necessary in the power range to add fuel to the basic air-fuel mixture requirements to keep cylinder-head temperatures in a safe range, the fuel mixture must be

[text] The best economy setting is established by running a series of curves through the cruise range, as shown in the graph in Figure 2-3, the low point (auto-lean) in the curve being the air-fuel mixture w

[caption] Figure 2-2. Power versus air-fuel mixture curve.

[caption] Figure 2-3. Specific fuel consumption curve.

[heading] Air-Fuel Mixtures

[text] Gasoline and other liquid fuels do not burn at all unless they are mixed with air. If the mixture is to burn properly within the engine cylinder, the ratio of air to fuel must be kept within a certain

[text] The composition of the air-fuel mixture is described by the mixture ratio. For example, a mixture with a ratio of 12 to 1



### Tables, as markdown

In [13]:
from IPython.display import Markdown, display

tables = [block for block in blocks if block.kind == "table"]
print(f"{len(tables)} tables")
for table in tables[:5]:
    display(Markdown(f"**page {table.page}**\n\n{table.text}"))

0 tables


### Figure captions

In [14]:
for caption in (block for block in blocks if block.kind == "caption"):
    print(f"p.{caption.page:>3}  {caption.text}")

p.  3  Figure 2-1. Air-fuel mixture curves.
p.  3  Figure 2-2. Power versus air-fuel mixture curve.
p.  3  Figure 2-3. Specific fuel consumption curve.
p.  5  Figure 2-4. Simple venturi.
p.  5  Figure 2-5. Side draft horizontal flow carburetor.
p.  6  Figure 2-6. Wide open throttle position.
p.  6  Figure 2-7. Fuel discharge.
p.  7  Figure 2-8. Needle valve and seat.
p.  7  Figure 2-9. Float chamber discharge nozzle and float.
p.  9  Figure 2-10. A float-type carburetor.
p. 10  Figure 2-11. Float chamber (bowl) with float removed. Figure 2-12. Main metering system.
p. 11  Figure 2-13. Air bleed principle.
p. 12  Figure 2-14. Throttle action in idle position.
p. 12  Figure 2-15. Idling system.
p. 12  Figure 2-16. Needle-type mixture control system.
p. 13  Figure 2-17. Back-suction-type mixture control system.
p. 13  Figure 2-18. Accelerating system.
p. 14  Figure 2-19. Accelerating pump shown in cutaway.
p. 14  Figure 2-21. A pressure operated economizer system. Figure 2-20. A needle-va

## Naive vs table-aware

The naive parse is the ablation baseline: plain page text, tables flattened
into the prose. Compare the same page both ways.

In [15]:
naive_blocks = parse.parse_pdf(pdf_path, table_aware=False)
table_page = tables[0].page if tables else PAGE

print("=== naive ===")
print(next(block.text for block in naive_blocks if block.page == table_page)[:1500])
print("\n=== table-aware ===")
print(parse.to_markdown([block for block in blocks if block.page == table_page])[:1500])

=== naive ===
2-3
Typical Air-Fuel Mixture Curve—Float-Type Carburetor
Rich
F/A
Lean
Takeoff
Maximum cruise
Minimum cruise
Auto rich
Auto lean
Climb
Idle
Typical Air-Fuel Mixture Curve—Pressure Injection Carburetor
Airflow in lb/hr
Low 
                                                                 High
Rich
F/A
Lean
Idle
Takeoff
Maximum cruise
Minimum cruise
Rich
Manual
Lean
Climb
Airflow in lb/hr
Low 
                                                                 High
Figure 2-1. Air-fuel mixture curves.
Air-Fuel Mixture
Lean 
 
 
 
                Rich
Power
High
Brake
HP
Low
Idle
Takeoff
Auto lean
Auto rich
Figure 2-2. Power versus air-fuel mixture curve.
Air-Fuel Mixture
Lean 
 
 
 
                Rich
Specific fuel consumption
High
Low
Auto lean
Auto rich
Figure 2-3. Specific fuel consumption curve. 
takeoff speed. [Figure 2-2] Since it is necessary in the power 
range to add fuel to the basic air-fuel mixture requirements 
to keep cylinder-head temperatures in a safe range,

## Save the parse for reading in full

In [ ]:
PARSED_DIR = Path("repo/data/parsed")
PARSED_DIR.mkdir(parents=True, exist_ok=True)

markdown_path = PARSED_DIR / f"{pdf_path.stem}.md"
markdown_path.write_text(parse.to_markdown(blocks))
print(f"wrote {markdown_path} ({markdown_path.stat().st_size / 1024:.0f} kB)")

wrote repo/data/parsed/ch10.md (230 kB)


## Whole corpus

In [16]:
corpus_blocks = parse.parse_corpus(RAW_DIR)
by_chapter = Counter((block.chapter, block.kind) for block in corpus_blocks)
for chapter in sorted({block.chapter for block in corpus_blocks}):
    counts = {kind: by_chapter[(chapter, kind)] for kind in ("text", "table", "caption")}
    print(f"chapter {chapter:>2}: {counts}")

chapter  1: {'text': 524, 'table': 2, 'caption': 85}
chapter  2: {'text': 354, 'table': 0, 'caption': 64}
chapter  3: {'text': 157, 'table': 2, 'caption': 58}
chapter  4: {'text': 594, 'table': 7, 'caption': 139}
chapter  5: {'text': 79, 'table': 3, 'caption': 27}
chapter  6: {'text': 285, 'table': 2, 'caption': 60}
chapter  7: {'text': 305, 'table': 0, 'caption': 56}
chapter  8: {'text': 233, 'table': 1, 'caption': 23}
chapter  9: {'text': 162, 'table': 0, 'caption': 28}
chapter 10: {'text': 648, 'table': 6, 'caption': 83}
chapter 11: {'text': 253, 'table': 4, 'caption': 34}


# Testing

In [17]:
from industrial_agent.retrieval.index import load_index
from industrial_agent.retrieval.search import search

index = load_index("/content/drive/MyDrive/faa_powerplant/index")

def ask(question, turbine_only=True):
    for hit in search(question, index=index, turbine_only=turbine_only):
        print(f"{hit.score:.2f}  {hit.chunk.citation}  {hit.chunk.section} > {hit.chunk.subsection}")
        print("      ", hit.chunk.text.split("\n\n", 1)[1][:200], "\n")

#ask("What are the best practices when inspecting turbine blades?")

In [ ]:
Q = "How do I gap a spark plug?"
print(f"--- Q: {Q} ---")
for hit in search(f"{Q}", index=index):
    print(f"{hit.score:.2f}  {hit.chunk.citation}  {hit.chunk.text[:]!r}")

--- Q: How do I gap a spark plug? ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

0.00  Ch. 10, pp. 10-49-50  'Combustion Section Inspection > Inspection & Repair of Combustion Chambers\n\nInspect the combustion chambers and covers for cracks by using visible dye or fluorescent penetrant inspection method. Any cracks, nicks, or dents are usually cause for rejecting the component. Inspect the covers, noting particularly the area around the fuel drain bosses for any pits or corrosion. When repairing the combustion chamber liner, the procedures given in the appropriate engine manufacturer’s overhaul instruction manual should be followed. If there is doubt that the liner is serviceable, it should be replaced.\n\nCombustion chambers should be replaced or repaired if two cracks are progressing from a free edge so that their meeting is imminent and could allow a piece of metal that could cause turbine damage to break loose. Separate cracks in the baffle are acceptable. Cracks in the cone are rare but, at any location on this component, is cause for rejection of the liner. 

In [ ]:
Q = "What is the torque for the CFM56 fan blade bolts?"
print(f"--- Q: {Q} ---")
for hit in search(f"{Q}", index=index):
    print(f"{hit.score:.2f}  {hit.chunk.citation}  {hit.chunk.text[:]!r}")

--- Q: What is the torque for the CFM56 fan blade bolts? ---
0.00  Ch. 10, p. 10-45  'Turbine Engine Maintenance > Causes of Blade Damage\n\nLoose objects often enter an engine either accidentally or through carelessness. Foreign object damage (FOD), such as pencils, tools, and flashlights, are often drawn into the engine and can cause damage to the fan blades. [Figure 10-59] Do not carry any objects in pockets when working around operational turbine engines.\n\nA compressor rotor can be damaged beyond repair by tools that are left in the air intake, where they are drawn into the engine on subsequent starts. A simple solution to the problem is to check the tools against a tool checklist. Prior to starting a turbine engine, make a minute inspection of engine inlet ducts to assure that items, such as nuts, bolts, lock wire, or tools, were not left there after work had been performed.\n\nFigure 10-60 shows some examples of blade damage to an axial-flow engine. The descriptions and possibl

In [ ]:
for Q, turbine_only in [("Engine has high EGT at takeoff power. What should I check?", True),
                        ("How do I gap a spark plug?", False)]:
    print(f"--- Q: {Q}  (turbine_only={turbine_only}) ---")
    for hit in search(Q, index=index, turbine_only=turbine_only):
        print(f"{hit.score:.3f}  {hit.chunk.citation}  {hit.chunk.engine_type}  {hit.chunk.section}")

--- Q: Engine has high EGT at takeoff power. What should I check?  (turbine_only=True) ---
0.739  Ch. 10, p. 10-54  turbine  Engine Ratings
0.579  Ch. 10, p. 10-60  turbine  Turbine Engine Calibration & Testing
0.402  Ch. 10, p. 10-59  turbine  Turbine Engine Calibration & Testing
0.211  Ch. 10, pp. 10-59-61  turbine  Turbine Engine Calibration & Testing
0.146  Ch. 10, p. 10-55  turbine  Turbine Engine Instruments
--- Q: How do I gap a spark plug?  (turbine_only=False) ---
0.260  Ch. 10, p. 10-36  reciprocating  Engine Troubleshooting
0.160  Ch. 10, p. 10-36  reciprocating  Engine Troubleshooting
0.045  Ch. 10, p. 10-27  reciprocating  Reciprocating Engine Operation
0.011  Ch. 10, p. 10-44  reciprocating  Cold Cylinder Check
0.010  Ch. 10, p. 10-13  reciprocating  Dimensional Inspection


In [18]:
Q = "I'm installing a carburetor which has some plugs on it, what do I do with the plugs?" # 04_amtp_ch2, pg 28, first line of installing a carburetor
print(f"--- Q: {Q} ---")
for hit in search(f"{Q}", index=index):
    print(f"{hit.score:.2f}  {hit.chunk.citation}  {hit.chunk.text[:]!r}")

--- Q: I'm installing a carburetor which has some plugs on it, what do I do with the plugs? ---


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (15 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

0.00  Ch. 10, p. 10-60  'Turbine Engine Calibration & Testing > Turbine Engine Analyzer Uses\n\n| Indicated Malfunction | Probable Causes | Suggested Action |\n| --- | --- | --- |\n| Note: Engines with damage in turbine section may have tendency to hang up during starting. | • If only exhaust gas temperature is high, other parameters normal, the problem may be thermocouple leads or instrument. | • Re-calibrate exhaust gas temperature instrumentation. |\n| Engine vibrates throughout rpm range, but indicated amplitude reduces as rpm is reduced. | • Turbine damage. | • Check turbine as outlined in preceding item. |\n| Engine vibrates at high rpm and fuel flow when compared to constant engine pressure ratio. | • Damage in compressor section. | • Check compressor section for damage. |\n| Engine vibrates throughout rpm range, but is more pronounced in cruise or idle rpm range. | • Engine-mounted accessory such as constant-speed drive, generator, hydraulic pump, etc. | • Check each component 

# Gradio web app test

In [ ]:
from industrial_agent.retrieval.app import launch

launch(index_dir="/content/drive/MyDrive/faa_powerplant/index",
       queries_path="/content/drive/MyDrive/faa_powerplant/queries.jsonl")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://546fc1284122ec7682.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Gradio Blocks instance: 9 backend functions
-------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x7daee6f534d0>
 |-<gradio.components.checkbox.Checkbox object at 0x7daee6f538c0>
 |-<gradio.components.checkbox.Checkbox object at 0x7daee5dedd10>
 |-<gradio.components.checkbox.Checkbox object at 0x7daee5dede50>
 |-<gradio.components.slider.Slider object at 0x7daee6f53a10>
 |-<gradio.components.slider.Slider object at 0x7daee5dee210>
 |-<gradio.components.slider.Slider object at 0x7daee5dee350>
 outputs:
 |-<gradio.components.markdown.Markdown object at 0x7daee5dee5d0>
 |-<gradio.components.dataframe.Dataframe object at 0x7daee6f53b60>
 |-<gradio.components.markdown.Markdown object at 0x7daee6f00410>
 |-<gradio.components.dropdown.Dropdown object at 0x7daee5db17f0>
 |-<gradio.components.state.State object at 0x7daee6f53230>
 |-<gradio.components.markdown.Markdown object at 0x7daee7253110>
fn_index=1
 inputs:
 |-<gradio.components.

In [ ]:
!cd repo && git pull

remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 14 (delta 10), reused 10 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (14/14), 12.79 KiB | 1.07 MiB/s, done.
From https://github.com/Cambie26/industrial_agent
   fcbf81e..13578f8  feature/maintenance-docs-rag -> origin/feature/maintenance-docs-rag
Updating fcbf81e..13578f8
Fast-forward
 notebooks/rag_dev.ipynb                  | 924 ++++++++++++++++++++++++++++++-
 pyproject.toml                           |   3 +-
 src/industrial_agent/retrieval/app.py    | 261 +++++++++
 src/industrial_agent/retrieval/search.py |  35 ++
 tests/test_retrieval.py                  | 100 ++++
 5 files changed, 1309 insertions(+), 14 deletions(-)
 create mode 100644 src/industrial_agent/retrieval/app.py


In [ ]:
%pip install -q -e "./repo[rag]"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for industrial-agent (pyproject.toml) ... done
